In [1]:
print("hello world")

hello world


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent          # notebook lives in notebooks/
RAW = ROOT / "data" / "raw"
print(ROOT)
print([f.name for f in RAW.iterdir()])

c:\Users\2409\Documents\Noob DEV\Milestone project 3 - new\parts-reorder-optimiser
['online_retail_II.csv']


Load / shape /columns data types

In [4]:
df = pd.read_csv(RAW / "online_retail_II.csv")
print(df.shape)
print(df.dtypes)
df.head(10)

(1067371, 8)
Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


row count

In [5]:
print("Rows:", len(df))

Rows: 1067371


missing values 

In [6]:
print(df.isna().sum())

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


Duplicate Values 

In [7]:
print("\nExact duplicate rows:", df.duplicated().sum())


Exact duplicate rows: 34335


unique Value Count

In [8]:
for col in df.columns:
    print(f"  {col:15} {df[col].nunique():>8,}")

  Invoice           53,628
  StockCode          5,305
  Description        5,698
  Quantity           1,057
  InvoiceDate       47,635
  Price              2,807
  Customer ID        5,942
  Country               43


Missing customers are fine because we forecast products not people, but the duplicates must be removed since counting the same sale twice would inflate demand

Time range 

In [9]:
date_col = "InvoiceDate"
df[date_col] = pd.to_datetime(df[date_col])

print("From:", df[date_col].min())
print("To  :", df[date_col].max())
print("Span days:", (df[date_col].max() - df[date_col].min()).days)

monthly = df.set_index(date_col).resample("ME").size()
print("\nRows per month:")
print(monthly)

From: 2009-12-01 07:45:00
To  : 2011-12-09 12:50:00
Span days: 738

Rows per month:
InvoiceDate
2009-12-31    45228
2010-01-31    31555
2010-02-28    29388
2010-03-31    41511
2010-04-30    34057
2010-05-31    35323
2010-06-30    39983
2010-07-31    33383
2010-08-31    33306
2010-09-30    42091
2010-10-31    59098
2010-11-30    78015
2010-12-31    65004
2011-01-31    35147
2011-02-28    27707
2011-03-31    36748
2011-04-30    29916
2011-05-31    37030
2011-06-30    36874
2011-07-31    39518
2011-08-31    35284
2011-09-30    50226
2011-10-31    60742
2011-11-30    84711
2011-12-31    25526
Freq: ME, dtype: int64


Product code shapes

In [10]:
codes = df["StockCode"].astype(str)

profile = pd.DataFrame({
    "length": codes.str.len(),
    "all_digits": codes.str.fullmatch(r"\d+"),
    "digits_then_letters": codes.str.fullmatch(r"\d+[A-Za-z]+"),
})

print("Code length distribution:")
print(profile["length"].value_counts().sort_index())
print("\nAll digits          :", profile["all_digits"].sum())
print("Digits then letters :", profile["digits_then_letters"].sum())

neither = codes[~profile["all_digits"] & ~profile["digits_then_letters"]]
print("\nNeither pattern:", neither.nunique(), "unique codes")
print(neither.value_counts().head(30))

Code length distribution:
length
1       1713
2        283
3       1446
4       2158
5     932385
6     127590
7       1393
8        127
9         74
12       202
Name: count, dtype: int64

All digits          : 932385
Digits then letters : 128892

Neither pattern: 63 unique codes
StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14
gift_0001_50       8
gift_0001_40       7
DCGS0069           6
B                  6
DCGS0004           5
m                  5
gift_0001_80       4
DCGS0072           4
DCGS0066N          4
DCGS0068           3
Name: count, dtype: int64



Quantity

In [11]:
q = df["Quantity"]
print(q.describe())
print("\nnegative:", (q < 0).sum(), " zero:", (q == 0).sum(), " positive:", (q > 0).sum())

neg = df[q < 0].copy()
neg["invoice_starts_C"] = neg["Invoice"].astype(str).str.upper().str.startswith("C")
print("\nNegative rows by invoice prefix:")
print(neg["invoice_starts_C"].value_counts())

print("\nSample where invoice does NOT start with C:")
print(neg.loc[~neg["invoice_starts_C"]].head(15))

count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64

negative: 22950  zero: 0  positive: 1044421

Negative rows by invoice prefix:
invoice_starts_C
True     19493
False     3457
Name: count, dtype: int64

Sample where invoice does NOT start with C:
     Invoice StockCode      Description  Quantity         InvoiceDate  Price  Customer ID         Country  invoice_starts_C
263   489464     21733     85123a mixed       -96 2009-12-01 10:52:00    0.0          NaN  United Kingdom             False
283   489463     71477            short      -240 2009-12-01 10:52:00    0.0          NaN  United Kingdom             False
284   489467    85123A      21733 mixed      -192 2009-12-01 10:53:00    0.0          NaN  United Kingdom             False
470   489521     21646              NaN       -50 2009-12-01 11:44:00    0.0          NaN  United Ki

Price

In [12]:
p = df["Price"]
print(p.describe())
print("\nzero:", (p == 0).sum(), " negative:", (p < 0).sum())

print("\nDescriptions on zero-price rows:")
print(df.loc[p == 0, "Description"].value_counts(dropna=False).head(15))

print("\nNegative price rows:")
print(df.loc[p < 0])

count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64

zero: 6202  negative: 5

Descriptions on zero-price rows:
Description
NaN                             4382
check                            162
?                                 92
damages                           84
damaged                           81
found                             28
missing                           27
sold as set on dotcom             20
Damaged                           17
adjustment                        16
OWL DOORSTOP                      15
POLYESTER FILLER PAD 45x45cm      12
dotcom                            12
amazon                            11
POLYESTER FILLER PAD 40x40cm      10
Name: count, dtype: int64

Negative price rows:
        Invoice StockCode      Description  Quantity         InvoiceDate     Price  Customer ID         Country
179403  A5

In [13]:
df["line_value"] = df["Quantity"] * df["Price"]

per_sku = (df.groupby("StockCode")
             .agg(rows=("line_value", "size"),
                  total_qty=("Quantity", "sum"),
                  total_value=("line_value", "sum"))
             .sort_values("total_value", ascending=False))

print("Top 20 by value (UNCLEANED — junk codes included):")
print(per_sku.head(20))
print("\nBottom 10:")
print(per_sku.tail(10))

Top 20 by value (UNCLEANED — junk codes included):
           rows  total_qty  total_value
StockCode                              
22423      4424      25764    327813.65
DOT        1446       2938    322647.47
85123A     5829      96066    253720.02
85099B     4216      95739    181278.51
47566      2768      27291    147948.50
84879      2960      80705    131413.85
22086      2217      35925    121662.14
POST       2122      10108    112341.00
79321      1222      16651     84854.16
22197      2549      79363     80300.07
22386      2347      39358     76244.93
84347       849      13148     73814.72
20725      3259      40011     70909.10
21137       661      19906     69247.18
85099F     1916      35853     67976.57
48138      1716      10126     67398.22
20685      1786       9889     67219.08
23084      1067      30646     66756.59
22114      1711      12312     62722.55
82484      1235      10999     62517.02

Bottom 10:
              rows  total_qty  total_value
StockCode     

1	Keep StockCode matching ^\d{5}[A-Za-z]*$



2	Keep Quantity > 0


3	Keep Price > 0


4	Drop exact duplicates


5	Rank by value after rules 1–4

In [15]:
pattern = r"^\d{5}[A-Za-z]*$"
codes = df["StockCode"].astype(str)

kept = codes[codes.str.fullmatch(pattern)].nunique()
dropped = sorted(codes[~codes.str.fullmatch(pattern)].unique())

print("SKUs kept   :", kept)
print("SKUs dropped:", len(dropped))
print(dropped)

SKUs kept   : 5242
SKUs dropped: 63
['47503J ', 'ADJUST', 'ADJUST2', 'AMAZONFEE', 'B', 'BANK CHARGES', 'C2', 'C3', 'CRUK', 'D', 'DCGS0003', 'DCGS0004', 'DCGS0006', 'DCGS0016', 'DCGS0027', 'DCGS0036', 'DCGS0037', 'DCGS0039', 'DCGS0041', 'DCGS0044', 'DCGS0053', 'DCGS0055', 'DCGS0056', 'DCGS0057', 'DCGS0058', 'DCGS0059', 'DCGS0060', 'DCGS0062', 'DCGS0066N', 'DCGS0066P', 'DCGS0067', 'DCGS0068', 'DCGS0069', 'DCGS0070', 'DCGS0071', 'DCGS0072', 'DCGS0073', 'DCGS0074', 'DCGS0075', 'DCGS0076', 'DCGSLBOY', 'DCGSLGIRL', 'DCGSSBOY', 'DCGSSGIRL', 'DOT', 'GIFT', 'M', 'PADS', 'POST', 'S', 'SP1002', 'TEST001', 'TEST002', 'gift_0001_10', 'gift_0001_20', 'gift_0001_30', 'gift_0001_40', 'gift_0001_50', 'gift_0001_60', 'gift_0001_70', 'gift_0001_80', 'gift_0001_90', 'm']


In [16]:
suspects = ["PADS", "SP1002"]
dcgs = [c for c in dropped if str(c).upper().startswith("DCGS")]

check = df[df["StockCode"].astype(str).isin(suspects + dcgs)]

summary = (check.assign(value=check["Quantity"] * check["Price"])
                .groupby("StockCode")
                .agg(rows=("value", "size"),
                     qty=("Quantity", "sum"),
                     value=("value", "sum"),
                     desc=("Description", lambda s: s.dropna().unique()[:2]))
                .sort_values("value", ascending=False))

print(summary.to_string())

           rows  qty    value                                desc
StockCode                                                        
DCGSSGIRL    25  191  295.630           [update, GIRLS PARTY BAG]
DCGS0076     15   18  292.090      [SUNJAR LED NIGHT NIGHT LIGHT]
DCGSSBOY     23   87  251.350            [update, BOYS PARTY BAG]
DCGS0069      6    0   80.310       [OOH LA LA DOGS COLLAR, ebay]
DCGS0004      5    2   67.940        [HAYNES CAMPER SHOULDER BAG]
DCGS0058     31   43   34.820                  [MISO PRETTY  GUM]
DCGS0003     14    6   32.590         [BOXED GLASS ASHTRAY, ebay]
DCGS0070      3   -6   25.440             [CAMOUFLAGE DOG COLLAR]
DCGS0072      4    3   20.680            [CAT CAMOUFLAGUE COLLAR]
DCGS0066N     4    4   17.300           [NAVY CUDDLES DOG HOODIE]
DCGS0068      3   -8   17.120           [DOGS NIGHT COLLAR, ebay]
SP1002        3  -22   14.750            [KID'S CHALKBOARD/EASEL]
DCGS0037      2    0   12.720                [KEY-RING CORKSCREW]
DCGS0075  